# 03 — Model Evaluation

Evaluasi model CNN-BiLSTM tersimpan pada test split menggunakan pipeline 45 fitur + velocity + acceleration.

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from scipy.ndimage import gaussian_filter1d

SEED_VALUE = 42
np.random.seed(SEED_VALUE)
tf.random.set_seed(SEED_VALUE)

PATH_DATASET_ROOT = "data/extracted"
MODEL_PATH = "assets/kasus7_murni_best.keras"
SCALER_MEDIAN_PATH = "assets/scaler_median_7.npy"
SCALER_IQR_PATH = "assets/scaler_iqr_7.npy"
CLASSES_LIST = ["jumpingjack", "lunges", "pushups", "squat", "Situp", "idle"]
SEQ_LEN = 30


## 1. Recreate Test Features

In [ ]:
def load_test_data(dataset_root=PATH_DATASET_ROOT):
    features, labels = [], []
    class_map = {c: i for i, c in enumerate(CLASSES_LIST)}
    split_path = os.path.join(dataset_root, "test")

    for class_name in CLASSES_LIST:
        class_dir = os.path.join(split_path, class_name)
        if not os.path.exists(class_dir):
            continue
        files = [os.path.join(class_dir, f) for f in sorted(os.listdir(class_dir)) if f.endswith(".npy")]
        for fpath in files:
            data = np.load(fpath)
            if data.shape[0] < SEQ_LEN:
                continue
            for i in range(0, data.shape[0] - SEQ_LEN + 1, SEQ_LEN):
                window = data[i:i + SEQ_LEN, :45].copy()
                smoothed = np.zeros_like(window)
                for j in range(window.shape[1]):
                    smoothed[:, j] = gaussian_filter1d(window[:, j], sigma=0.3)
                velocity = np.diff(smoothed, axis=0, prepend=smoothed[0:1])
                acceleration = np.diff(velocity, axis=0, prepend=velocity[0:1])
                features.append(np.concatenate([window, velocity, acceleration], axis=1))
                labels.append(class_map[class_name])
    return np.asarray(features, dtype=np.float32), np.asarray(labels, dtype=np.int32)


## 2. Temporal Ensemble

In [ ]:
def temporal_ensemble(y_pred_probs, window_size=5):
    smoothed = np.copy(y_pred_probs)
    for i in range(len(y_pred_probs)):
        start = max(0, i - window_size + 1)
        smoothed[i] = np.mean(y_pred_probs[start:i + 1], axis=0)
    return np.argmax(smoothed, axis=1)


## 3. Model Evaluation

In [ ]:
X_test, y_test = load_test_data()
median = np.load(SCALER_MEDIAN_PATH).squeeze()
iqr = np.load(SCALER_IQR_PATH).squeeze()
X_test = (X_test - median) / (iqr + 1e-8)

model = tf.keras.models.load_model(MODEL_PATH, compile=False)
y_pred_probs = model.predict(X_test, verbose=0)
y_pred_raw = np.argmax(y_pred_probs, axis=1)
y_pred_ensemble = temporal_ensemble(y_pred_probs, window_size=5)

print(f"Raw accuracy: {accuracy_score(y_test, y_pred_raw) * 100:.2f}%")
print(f"Temporal ensemble accuracy: {accuracy_score(y_test, y_pred_ensemble) * 100:.2f}%")
print("\nClassification report (temporal ensemble):")
print(classification_report(y_test, y_pred_ensemble, target_names=CLASSES_LIST))


## 4. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_ensemble)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES_LIST, yticklabels=CLASSES_LIST)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — CNN-BiLSTM")
plt.tight_layout()
plt.show()


## Catatan

Hasil numerik harus diverifikasi dengan hasil final penelitian sebelum dijadikan headline portfolio.